# Resultados con fotos reales — Detector de puntos calientes (ASICAI)

**Autor:** Christian Masaquiza Jerez · **Dron:** DJI Mavic 3T (térmica 640×512) · **Complementa a:** `experimentos.ipynb` (datos sintéticos)

**Objetivo:** medir cómo se comporta el detector baseline con fotos térmicas reales autorizadas, compararlo (si se hizo) con YOLO y documentar límites.

> ⚠️ **Confidencialidad.** Las fotos son de clientes de ASICAI. Este notebook **no muestra imágenes** por defecto (`MOSTRAR_IMAGENES = False`). Si las activas, **no hagas commit con las salidas**: ejecuta `nbstripout resultados_reales.ipynb` antes. Las tablas numéricas sí se pueden compartir.

> Alcance: detecta zonas más calientes que su entorno para **apoyar** la revisión de un técnico. **No diagnostica fallas.**

## 1. Preparación y configuración

In [ ]:
import sys, os, csv
from pathlib import Path
sys.path.insert(0, os.path.abspath("src"))   # ejecutar desde la raíz del proyecto (asicai-poc)

import numpy as np
import cv2
import matplotlib.pyplot as plt

import real_tools
from eval_real import evaluate
from yolo_io import read_labels

# ---- Configuración (ajusta aquí) ----
OUT = Path("data/real_private")      # carpeta generada por real_baseline.py
TEMP_DIR = OUT / "temp_c_tiff"       # TIFF de °C de dji-thermal-cli (si existe, se usan; si no, se usa la caché o el proxy)
SDK_EXE = None                       # alternativa: ruta a dji_irp(.exe) del SDK oficial, solo si lo usaste
MOSTRAR_IMAGENES = False             # True solo en tu PC y sin hacer commit de las salidas
BLIND_N = None                       # [completar] cuántas fotos revisaste en modo --blind
PARAMS_BASE = dict(percentile=97.5, std_k=2.2, min_area_px=20)

(OUT / "tablas").mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.dpi"] = 100
assert (OUT / "index.csv").exists(), "Falta data/real_private/index.csv: corre primero src/real_baseline.py"
print("Listo")

## 2. Inventario: ¿con qué datos cuento?
Fotos térmicas (`_T`), escenas (fotos tomadas con pocos segundos de diferencia, casi iguales entre sí) y tamaño.
Las **escenas** importan más que las fotos: 30 fotos de 3 escenas equivalen a muy poca variedad.

In [ ]:
with open(OUT / "index.csv", encoding="utf-8") as f:
    inv = list(csv.DictReader(f))

n_fotos = len(inv)
grupos = {}
for r in inv:
    grupos.setdefault(int(r["group"]), []).append(r)
dims = {(r["width"], r["height"]) for r in inv}

print(f"Fotos térmicas: {n_fotos}")
print(f"Escenas (grupos): {len(grupos)}")
print(f"Resoluciones encontradas: {dims}  (la térmica del Mavic 3T es 640x512)")
print(f"Fotos con par visual (_V): {sum(1 for r in inv if r['v_path'])}")
print(f"Rango de fechas: {min(r['datetime'] for r in inv)[:10]} → {max(r['datetime'] for r in inv)[:10]}")

plt.figure(figsize=(8, 3))
plt.bar(list(grupos), [len(v) for v in grupos.values()], color="#888")
plt.xlabel("escena (grupo)"); plt.ylabel("fotos"); plt.title("Fotos por escena"); plt.show()

**Mi interpretación (completar con tus palabras):**
- ¿Cuántas escenas distintas hay realmente? ¿Es suficiente variedad para entrenar un modelo?
- ¿Qué tipos de equipo aparecen? ¿Hay paletas/condiciones distintas entre sesiones?

## 3. Baseline sobre todas las fotos (sin etiquetas)
Aquí **no hay métricas de acierto** todavía, solo comportamiento: cuántas detecciones por foto.
Un detector por percentil siempre tiende a marcar "lo más caliente" de cada foto; las fotos **sin** detecciones y las que tienen **muchas** son pistas de falsos positivos.

In [ ]:
with open(OUT / "detections.csv", encoding="utf-8") as f:
    det = list(csv.DictReader(f))

tipo = {r["tipo_dato"] for r in det}
print("Tipo de dato:", tipo)
if "proxy" in tipo:
    print("⚠️ 'proxy' = brillo del falso color, NO temperatura. Los resultados son orientativos.")

n_det = np.array([int(r["n_detecciones"]) for r in det])
print(f"Fotos sin detecciones: {(n_det == 0).sum()} de {len(n_det)}")
print(f"Detecciones por foto: media {n_det.mean():.2f}, máx {n_det.max()}")

plt.figure(figsize=(7, 3))
plt.hist(n_det, bins=range(0, n_det.max() + 2), color="#888", align="left")
plt.xlabel("detecciones por foto"); plt.ylabel("fotos"); plt.title("Distribución de detecciones (baseline)"); plt.show()

with open(OUT / "tablas" / "resumen_detecciones.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["fotos", "sin_detecciones", "media_det", "max_det", "tipo_dato"])
    w.writerow([len(n_det), int((n_det == 0).sum()), round(float(n_det.mean()), 2), int(n_det.max()), "/".join(sorted(tipo))])

In [ ]:
# Galería de comprobación visual (solo en tu PC; ver advertencia de confidencialidad)
if MOSTRAR_IMAGENES:
    muestra = sorted(det, key=lambda r: -int(r["n_detecciones"]))[:6]
    fig, ax = plt.subplots(2, 3, figsize=(12, 7))
    for a, r in zip(ax.ravel(), muestra):
        im = cv2.imread(str(OUT / "overlays" / f"{r['stem']}.jpg"))
        a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.axis("off")
        a.set_title(f"{r['stem']} · {r['n_detecciones']} det.", fontsize=8)
    plt.show()
else:
    print("Imágenes ocultas (MOSTRAR_IMAGENES = False).")

## 4. Ground truth: ¿cuántas fotos revisé a mano?
Las métricas reales **solo** se calculan sobre fotos revisadas con `review_labels.py`.
Criterio de etiqueta: "zona visiblemente más caliente que su entorno que un técnico querría revisar" — **[completar: ¿lo validó alguien de ASICAI?]**

In [ ]:
rev_file = OUT / "reviewed.txt"
assert rev_file.exists(), "Aún no hay fotos revisadas: corre src/review_labels.py"
reviewed = rev_file.read_text().split()
gt_dir = OUT / "labels_reviewed"
n_cajas = sum(len(read_labels(gt_dir / f"{s}.txt")) for s in reviewed)
vacias = sum(1 for s in reviewed if not read_labels(gt_dir / f"{s}.txt"))
grupo_de = {r["stem"]: int(r["group"]) for r in inv}

print(f"Fotos revisadas: {len(reviewed)} de {n_fotos} | escenas cubiertas: {len({grupo_de[s] for s in reviewed})}")
print(f"Cajas reales (GT): {n_cajas} | fotos revisadas sin ninguna caja (negativos): {vacias}")
print(f"Revisadas en modo ciego: {BLIND_N if BLIND_N is not None else '[completar BLIND_N]'}")

val_file = OUT / "val_stems.txt"
if val_file.exists():
    stems_val = val_file.read_text().split()
    stems_tune = [s for s in reviewed if s not in set(stems_val)]
else:
    stems_val, stems_tune = reviewed, reviewed
    print("⚠️ No hay val_stems.txt (corre make_yolo_dataset.py). Uso TODAS las revisadas para ajustar y para evaluar: la métrica saldrá optimista.")
print(f"Para ajustar parámetros: {len(stems_tune)} fotos | para reportar resultado final (validación): {len(stems_val)} fotos")

## 5. Métricas del baseline en fotos reales
Un acierto (TP) = caja predicha con IoU ≥ umbral frente a una caja real. En objetos de pocos píxeles el IoU es muy exigente, por eso se muestran varios umbrales.
Los parámetros se **ajustan en fotos de `stems_tune`** y el resultado final se reporta en `stems_val` (si no, estaría "haciendo trampa" al reportar sobre lo que usé para elegir).

In [ ]:
arrays = real_tools.cargar_arrays(OUT, reviewed, sdk_exe=SDK_EXE, temp_dir=TEMP_DIR)
print("Tipo de dato (según detections.csv):", tipo, "→ si dice proxy, vuelve a correr real_baseline.py con --temp-dir")

filas = []
for thr in (0.1, 0.3, 0.5):
    m = real_tools.eval_baseline(arrays, gt_dir, stems_val, iou=thr, **PARAMS_BASE)
    filas.append({"IoU": thr, **m})
    print(f"IoU≥{thr}: {m}")

with open(OUT / "tablas" / "baseline_validacion.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(filas[0])); w.writeheader(); w.writerows(filas)

## 6. Barrido de parámetros sobre fotos reales (equivalente a tus Experimentos A y B)
Se barre un parámetro dejando los demás en `PARAMS_BASE`. Se usan las fotos de ajuste (`stems_tune`), IoU ≥ 0.3.

In [ ]:
def barrido(nombre, valores, **fijos):
    res = []
    for v in valores:
        p = {**PARAMS_BASE, **fijos, nombre: v}
        res.append((v, real_tools.eval_baseline(arrays, gt_dir, stems_tune, iou=0.3, **p)))
    return res

def mostrar(nombre, res, log=False):
    print(f"{nombre:>12} {'precision':>10} {'recall':>8} {'FP':>5} {'FN':>5}")
    for v, r in res:
        print(f"{v:>12} {r['precision']:>10} {r['recall']:>8} {r['FP']:>5} {r['FN']:>5}")
    plt.figure(figsize=(7, 3.5))
    xs = [v for v, _ in res]
    plt.plot(xs, [r["precision"] for _, r in res], "o-", label="precisión")
    plt.plot(xs, [r["recall"] for _, r in res], "s-", label="recall")
    if log: plt.xscale("log")
    plt.xlabel(nombre); plt.ylim(0, 1.05); plt.grid(alpha=.3); plt.legend()
    plt.title(f"Fotos reales: efecto de {nombre}"); plt.show()
    with open(OUT / "tablas" / f"barrido_{nombre}.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow([nombre, "precision", "recall", "FP", "FN"])
        for v, r in res: w.writerow([v, r["precision"], r["recall"], r["FP"], r["FN"]])

res_k = barrido("std_k", [0.0, 0.5, 1.0, 1.5, 2.2, 3.0, 4.0], percentile=90)
mostrar("std_k", res_k)

In [ ]:
res_a = barrido("min_area_px", [5, 10, 20, 40, 80, 150, 300])
mostrar("min_area_px", res_a, log=True)

**Mi interpretación (completar con tus palabras):**
- ¿Dónde empieza a caer la precisión y dónde el recall en fotos reales? ¿Coincide con los sintéticos?
- Con la tabla de tamaño de píxel por distancia (informe de viabilidad §6): ¿qué distancia de vuelo corresponde a las áreas donde pierdes hotspots?
- ¿Qué configuración recomendarías para mantenimiento predictivo y por qué (priorizar no perder puntos vs. evitar falsas alarmas)?

### 6b. (Opcional) Mismo barrido en datos sintéticos, para comparar lado a lado
Usa tus módulos `thermal_simulator`, `hotspot_detector` y `evaluate`. Si no están, esta celda se omite.

In [ ]:
try:
    from thermal_simulator import generate_scene
    from hotspot_detector import detect_hotspots
    from evaluate import match_detections_to_gt, compute_metrics
    ESC = [generate_scene(n_hotspots=int(1 + i % 3), seed=2000 + i) for i in range(30)]
    def evaluar_sint(**p):
        M, FN, FP = [], [], []
        for im, g in ESC:
            m, fn, fp = match_detections_to_gt(detect_hotspots(im, **p), g)
            M += m; FN += fn; FP += fp
        return compute_metrics(M, FN, FP)
    print("Sintéticos, barrido de min_area_px (percentil 97.5, std_k 2.2):")
    for a in [5, 20, 60, 150, 300, 600]:
        r = evaluar_sint(percentile=97.5, std_k=2.2, min_area_px=a)
        print(f"  area {a:>4}: precision {r['precision']} recall {r['recall']}")
except Exception as e:
    print("Celda omitida:", type(e).__name__, e)

## 7. ¿Dónde falla? Errores por foto
Las fotos con más FP/FN son las que hay que **mirar** (en tu PC) para entender el porqué: reflejos, equipos calientes por diseño, bordes, paleta, distancia…

In [ ]:
peores = real_tools.errores_por_foto(arrays, gt_dir, stems_val, iou=0.3, **PARAMS_BASE)
print(f"{'foto':>22} {'TP':>3} {'FP':>3} {'FN':>3}")
for s, tp, fp, fn in peores[:10]:
    print(f"{s:>22} {tp:>3} {fp:>3} {fn:>3}")

**Errores típicos que observé (completar tras mirar las fotos en `overlays/`, sin pegar imágenes confidenciales):**
- Falsos positivos: …
- Falsos negativos: …

## 8. YOLO vs baseline (solo si hiciste el entrenamiento)

In [ ]:
pred_yolo = OUT / "pred_yolo"
if pred_yolo.exists():
    for thr in (0.1, 0.3, 0.5):
        b = real_tools.eval_baseline(arrays, gt_dir, stems_val, iou=thr, **PARAMS_BASE)
        y = evaluate(gt_dir, pred_yolo, stems_val, thr)
        print(f"IoU≥{thr}\n  baseline: {b}\n  YOLO    : {y}")
    print(f"\nTamaño de la validación: {len(stems_val)} fotos. Con tan pocas, la diferencia puede ser ruido.")
else:
    print("No hay pred_yolo/: paso opcional no realizado. (Es una conclusión válida: se documenta como pendiente.)")

## 9. Conclusiones y limitaciones
(Escríbelas tú con tus palabras al terminar. Apóyate en las tablas de `data/real_private/tablas/`.)

- **Qué aprendí con fotos reales vs sintéticas:** …
- **Configuración que recomendaría y por qué:** …
- **Limitaciones:** muestra de [N] fotos / [M] escenas; etiquetas hechas por [una persona]; criterio [validado / no validado] por ASICAI; dato [°C / proxy]; **sin diagnóstico de fallas**.
- **Siguiente paso:** …